In [ ]:
import os
import numpy as np
import pandas as pd
import commons as c

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.metrics import precision_score, recall_score, f1_score

# Boxplots

In [ ]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)
df_normal['nature'] = 'non-equivalent'

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)
df_equiv['nature'] = 'equivalent'


In [ ]:
df = pd.concat([df_equiv, df_normal], ignore_index=True)
df['true_label'] = np.where(df['true_label'] == True, "non-equivalent", "equivalent")
df['predicted_label'] = np.where(df['predicted_label'] == True, "non-equivalent", "equivalent")

In [ ]:
def print_box_plot(df, cat, file_name, type):

    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column    
    cat_mapping = {category: idx for idx, category in enumerate(cat_range)}
    df['cat_numeric'] = df[cat].map(cat_mapping)  # Map categories to numeric values

    # Add an offset to the x-axis values based on 'true_label' to spread the points
    label_offsets = {
        "equivalent": -0.1,   # Adjust this value as needed
        "non-equivalent": 0.1  # Adjust this value as needed
    }
    
    # Apply the offset to a new column for the x-axis
    df['x_offset'] = df['cat_numeric'] + df['nature'].map(label_offsets)

    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['nature'] = df['nature'].map(label_mapping)
    
    # Create the scatter plot with the adjusted x-axis values
    fig = px.box(#scatter
        df, 
        y=type, 
        x="x_offset", 
        color="nature", 
        category_orders={cat: cat_range},
        points=False
    ) 
    
    # Adjust layout for better visualization
    fig.update_layout(
        scattermode="group",
        xaxis=dict(
            title=cat, 
            categoryorder="array", 
            categoryarray=cat_range,
            tickvals=list(range(len(cat_range))),
            ticktext=cat_range
        ),
        yaxis_title="Distance between original and mutant",
        xaxis_title="Characteristic",
        legend_title_text="Expected value",
        boxgroupgap=0, 
        boxgap=0
    )

    # Save the figure
    output_folder = 'results/RQ1/boxplots/'
    os.makedirs(output_folder, exist_ok=True)
    c.setup_layout_and_save(fig, "Visualisation of the mutant detectability score for each metric", output_folder, file_name, yaxis_range=[0, 1])


In [ ]:
columns = ['Qubits_number', 'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates'] 
df_hw = df[df['hardware'] == 'kyiv']
selected_columns = df_hw[["metric", 'nature', 'ideal_distance']]
file_name = f'visu_ideal'
print_box_plot(selected_columns, "metric", file_name, 'ideal_distance')

for hw in c.hardware:
    df_hw = df[df['hardware'] == hw]
    selected_columns = df_hw[["metric", 'nature', 'noisy_distance']]
    file_name = f'visu_{hw}'
    print_box_plot(selected_columns, "metric", file_name, 'noisy_distance')
        

# F1, Precision and recall

In [ ]:
def get_scores(df):
    results = []
    
    for metric in c.metrics.keys():
        
        true_labels = df[f'Killed_I{metric}']
        predicted_labels = df[f'Killed_N{metric}']
        
        # Calculate precision, recall, and F1 score for the pair
        precision = precision_score(true_labels, predicted_labels, zero_division=0)
        recall = recall_score(true_labels, predicted_labels, zero_division=0)
        f1 = f1_score(true_labels, predicted_labels, zero_division=0)
        results.append((precision, recall, f1))
    
    scores_df = pd.DataFrame(results, columns=['Precision', 'Recall', 'F1 Score'], 
                             index=[f'{metric}' for metric in c.metrics])
    
    return scores_df.round(4)


In [ ]:
output_folder = 'results/RQ1/scores'
os.makedirs(output_folder, exist_ok=True)

scores = ["Precision", "Recall", "F1 Score"]

mutant_type = "balanced"

# Initialize dictionaries to store score DataFrames with (metric, threshold) index
dic_scores = {
    score: pd.DataFrame(columns=c.hardware, index=pd.MultiIndex.from_product(
        [[f'Metric {metric}' for metric in c.metrics.keys()], c.thresholds], names=["Metric", "Threshold"]
    ))
    for score in scores
}
    
for threshold in c.thresholds:
    for hw in c.hardware:
        csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
        df = pd.read_csv(csv_path)
        scores_df = get_scores(df)
            
        for score in scores:
            for metric in scores_df.index:
                dic_scores[score].at[(metric, threshold), hw] = scores_df.at[metric, score]

# Save each score DataFrame as a CSV
for score, df_score in dic_scores.items():
    
    output_path = os.path.join(output_folder, f'{mutant_type}_{score}.csv')
    df_score.to_csv(output_path)
    print(f"Saved {output_path}")


# Confusion matrices


In [ ]:
def confusion_matrix(df, col1, col2):
    # Create a confusion matrix DataFrame
    conf_matrix = pd.DataFrame(index=['True', 'False'], columns=['True', 'False'])
    
    # Calculate the count of each pair
    true_true = ((df[col1] == True) & (df[col2] == True)).sum()
    false_false = ((df[col1] == False) & (df[col2] == False)).sum()
    true_false = ((df[col1] == True) & (df[col2] == False)).sum()
    false_true = ((df[col1] == False) & (df[col2] == True)).sum()
    
    # Total number of rows
    total = len(df)
    
    # Calculate percentages
    conf_matrix.loc['True', 'True'] = (true_true / total) * 100
    conf_matrix.loc['False', 'False'] = (false_false / total) * 100
    conf_matrix.loc['True', 'False'] = (true_false / total) * 100
    conf_matrix.loc['False', 'True'] = (false_true / total) * 100
    
    # Ensure all values are numeric and handle any potential issues
    conf_matrix = conf_matrix.apply(pd.to_numeric, errors='coerce')  # Convert to numeric, coerce errors to NaN
    conf_matrix.fillna(0, inplace=True)  # Replace NaNs with 0 if there are any

    return conf_matrix

In [ ]:
# Define a function to create a heatmap with annotations
def create_heatmap(fig, data, row, col, showscale):
    fig.add_trace(
        go.Heatmap(
            z=data,
            text=data,  # Use the same data for annotations
            colorscale= [[0.0, '#eff3ff'], [0.05, '#9ecae1'],[0.1, '#6baed6'], [0.8, '#3182bd'], [1, '#08519c']],
            colorbar=dict(title='Scale'),
            zmin=0, zmax=100,
            showscale=showscale,
            texttemplate='%{text:.2f}',  # Format the text annotations
            textfont=dict(size=18)
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    fig.update_yaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    

In [ ]:
def print_confusion_matrices(threshold, mutant):
    # Create subplots with titles
    fig = make_subplots(
        rows=1, cols=6,
        subplot_titles=c.metric_names,
        x_title='Noisy', y_title='Ideal', horizontal_spacing=0.05
    )
    
    dataframes = []
    for hw in c.hardware:
        csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
        dataframes.append(pd.read_csv(csv_path))
    
    df_confusion = pd.concat(dataframes, ignore_index=True)
    
    for j, metric in enumerate(c.metrics):
        matrix = confusion_matrix(df_confusion, 'Killed_I' + metric, 'Killed_N' + metric)
        # Add heatmaps to subplots
        create_heatmap(fig, matrix, row=1, col=j+1, showscale=True)
    
    # Update layout and save as image
    fig.update_layout(
        title_text=f'Overall confusion matrix for {mutant} mutants with a threshold of {threshold}',
        height=400,
        width=2000,
        showlegend=False
    )
    
    #fig.show()
    fig.write_image(f"results/RQ1/confusion_matrices/{mutant}_{threshold}.png")

In [ ]:
output_folder = 'results/RQ1/confusion_matrices/'
os.makedirs(output_folder, exist_ok=True)

for threshold in c.thresholds:
    for mutant_type in c.mutant_types:
        print_confusion_matrices(threshold, mutant_type)